<a href="https://colab.research.google.com/github/yerlanbektenovwbs/YerlanBektenov-AnalyticsEngineering/blob/main/Assignment%201%20-%20Copy_of_AE_test_task_C_customers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=17M9oGaOj4KdKiDDH3oTibxJ9qBTSjp6n)

# Analytics Engineering: timed test

## **PLEASE FIRST SAVE A COPY OF THIS FILE! GO TO FILE > SAVE A COPY IN DRIVE.**
### This will mean that your work auto-saves and prevents any issues if you have internet or browser issues. Remember to work on the saved copy. It will be titled "Copy of AE_test_task ..."
<br>

## Task C: Customer payments

## How this test works

- You have **55 minutes to complete the work** with an additional 5 minutes at the end to upload the work. However, there is no end-point as such, you are completing part of what would likely be a longer task.
- This is individual work - please do not confer with anyone inside or outside the room.
- You may use **any AI tool** (for example, Gemini in Colab - which is probably the quickest route). You are **not marked on the quality of the code** it produces. You are marked on how well you control the work: what you look at, what you check, what you reject or correct and why, and what you could not verify.
- Therefore the quality of the AI used has no bearing on your marks, as you are **not marked on the quality of the code** it produces.
- Write in markdown cells as you go: your thinking, **each prompt you send (paste it as you send it)**, and the checks you make. Notes written afterwards are usually vaguer. If you are using Gemini as in built in the Notebook the prompts are **not** automatically saved - you still need to paste them into section 5 or alongside your code, **along with the checks you determine to make sure the AI code works**.
- Unfinished work is fine if your checking and reasoning are clear. Say what you did not get to.
- Please do not discuss this task with anyone who has not yet sat the test. Other groups have different briefs.

## Rubric

| Element | Marks |
| --- | --- |
| Question one | 100% |
| | |
| Application of analytics engineering techniques: applying methods from the module such as data cleaning, joining, etc. | 35% |
| Careful execution of code (whether generated by AI, from module resources (e.g. GitHub) or memory) and checking the code has worked.  | 35% |
| Analysis and interpretation of results: what do your checks show about the data and what steps should come next? | 30% |


<br>

## The data

You have eight tables from a real Brazilian online marketplace (Olist), covering orders from 2016 to 2018. A geolocation table exists in the full dataset but is not included in this test.

![](https://drive.google.com/uc?export=view&id=10zwOXJ1e9IpT58rhvxgeMqaUZXiqYzDJ)

| Name in this notebook | What it holds |
| --- | --- |
| `orders` | The core table: one record per order, with status and timestamps |
| `order_items` | The items inside each order |
| `payments` | How each order was paid |
| `reviews` | Customer reviews of orders |
| `customers` | Customer details and location |
| `products` | Product details, including category |
| `sellers` | Seller details and location |
| `category_translation` | Maps product category names to English |

**How the tables connect**

- `orders` to `order_items`, `payments` and `reviews` on `order_id`
- `orders` to `customers` on `customer_id`
- `order_items` to `products` on `product_id`
- `order_items` to `sellers` on `seller_id`
- `category_translation` to `products` on the product category name

Check the actual column names in the next cell before you write any joins.


In [1]:
# ---- Setup: run this cell first ----
import glob
import os

import duckdb
import pandas as pd

# Download the files for the test
DATA_URL = "https://github.com/MJMortensonWarwick/adminaeaiv/releases/download/testsession/brazilian-ecommerce.zip"    # link to a .zip containing the CSV files

DATA_DIR = "/content/olist"   # upload the CSV files here using the Files panel
  # Download only if no CSV files are there yet. The CSVs may sit inside a folder
  # within the zip; the search below looks in sub-folders too.
if DATA_URL and not glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True):
    import urllib.request
    import zipfile
    os.makedirs(DATA_DIR, exist_ok=True)
    urllib.request.urlretrieve(DATA_URL, "/content/olist.zip")
    zipfile.ZipFile("/content/olist.zip").extractall(DATA_DIR)

FILES = {
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

tables, missing = {}, []
for name, filename in FILES.items():
    found = glob.glob(os.path.join(DATA_DIR, "**", filename), recursive=True)
    if found:
        tables[name] = pd.read_csv(found[0])
    else:
        missing.append(filename)

if missing:
    found_csvs = sorted(os.path.basename(f) for f in glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
    raise FileNotFoundError(
        f"Missing files: {missing}. CSV files found under {DATA_DIR}: {found_csvs}. Ask your invigilator for help."
    )

# Make every table available as a DuckDB table, so you can use SQL or pandas.
con = duckdb.connect()
for name, df in tables.items():
    con.register(name, df)


def q(sql):
    """Run SQL against the tables above and return a pandas DataFrame."""
    return con.sql(sql).df()


for name, df in tables.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")
    print("   ", list(df.columns))

orders: 99,441 rows, 8 columns
    ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
order_items: 112,650 rows, 7 columns
    ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']
payments: 103,886 rows, 5 columns
    ['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']
reviews: 89,999 rows, 7 columns
    ['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']
customers: 99,441 rows, 5 columns
    ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']
products: 32,951 rows, 9 columns
    ['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'produc

## Examples: two ways to work with the tables

You can use SQL, pandas, or a mix of both. The next two cells show each, using only the tables and keys described above. Run them and change them to see what they do. You do not have to use them in your answer.


In [2]:
# Example 1: SQL
# q() runs SQL against any of the tables above and returns a pandas DataFrame.
from IPython.display import display

# Look at the first few rows of a table
display(q("SELECT * FROM orders LIMIT 5"))

# How many rows are there, and how many different order_id values?
display(q("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT order_id) AS n_distinct_orders FROM orders"))

# Join two tables on a key
display(q("""
    SELECT o.order_id, c.customer_id
    FROM orders AS o
    JOIN customers AS c ON o.customer_id = c.customer_id
    LIMIT 5
"""))


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


,n_rows,n_distinct_orders
0,99441,99441


,order_id,customer_id
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089
3,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c
4,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608


In [3]:
# Example 2: pandas
# Each table is also a pandas DataFrame, stored in the dictionary `tables`.
from IPython.display import display

orders = tables["orders"]
display(orders.head())        # first few rows
print(orders.shape)           # (number of rows, number of columns)
print(orders.dtypes)          # data type of each column

# The same join as in Example 1, using pandas
joined = tables["orders"].merge(tables["customers"], on="customer_id", how="inner")
display(joined[["order_id", "customer_id"]].head())

# You can mix the two: run SQL, then use pandas methods on the result
result = q("SELECT customer_id FROM orders")
print(result["customer_id"].nunique(), "different customer_id values in orders")


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


(99441, 8)
order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object


,order_id,customer_id
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c


99441 different customer_id values in orders


## Your brief

**From:** Finance

> We want to understand how customers pay. For each month, show the share of orders and the share of revenue by payment type, and the average number of instalments for credit card payments. Can we trust that the shares add up?

**What your final table should look like**

One table with columns for month, payment type, number of orders, revenue, share of month’s orders, share of month’s revenue, average instalments.

**Decisions you must make and state**

- An order can have several payment rows (vouchers combined with a card, for example). How will you assign a payment type to an order, or will you count at payment level instead?
- Which orders count, and what does "revenue" mean here?

**If you have time**

Show how many customers placed a second order within 90 days of their first.


## 1. Understand the data

*Look before you build. Write down what you find: what one row of each table represents, how the tables connect, and anything that looks odd or missing.*

In [4]:
for name, df in tables.items():
    print(f"========================================")
    print(f"Table: {name} | Total Rows: {df.shape[0]:,}")
    print(f"========================================")
    display(df.head(3))
    print("\n")

Table: orders | Total Rows: 99,441


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00




Table: order_items | Total Rows: 112,650


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.0,17.87




Table: payments | Total Rows: 103,886


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71


... (6 outputs left out)



Table: sellers | Total Rows: 3,095


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ




Table: category_translation | Total Rows: 71


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto


Task: For each month, show the share of **orders** and the share of **revenue** by **payment type**, and the average number of instalments for credit card payments.

Goal: One table with columns for **month**, **payment type**, **number of orders, revenue, share of month’s orders, share of month’s revenue, average instalments**.


AI promt: for each table in tables, loop through and print the table name, the number of rows in the table and the first three records

ORDERS: contains order_id (NUMBER OF ORDERS), order_status, order_purchase_timestamp (MONTH)

PAYMENTS: contains order_id, payment_values and payment_types
REVENUE, and installments

These two tables seem to contains most data (columns) needed to produce the final table (goal)

We need to count number of unique order_id. Ideally, this should equal to the number of rows. We alse need to check for missing data (and not include in our analysis there rows if we dont have id).

If order_id doesnt show duplicates, then for each order_status we have a unique order_id. Below we see that most orders have 'delivered' status, then shipped, cancelled, etc. For our analysis, we can focus on the first three, based on the fact that these represent most orders by far.

We can use order_purchase_timestamp to consolidate data by month as requested. We need to make sure the data type is stadardised and not missing.

We have 99,441 orders and 103,886 payment rows. We need to join the tables on oder_id.


In [6]:
#prompt: for order table, show number of rows for each order_status column

display(q("""
    SELECT order_status, COUNT(*) AS count_of_orders
    FROM orders
    GROUP BY order_status
    ORDER BY count_of_orders DESC
"""))

,order_status,count_of_orders
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


In [7]:
display(q("""
    SELECT
        'orders' AS table_name,
        COUNT(*) AS total_rows,
        COUNT(DISTINCT order_id) AS unique_orders
    FROM orders

    UNION ALL

    SELECT
        'payments' AS table_name,
        COUNT(*) AS total_rows,
        COUNT(DISTINCT order_id) AS unique_orders
    FROM payments
"""))

,table_name,total_rows,unique_orders
0,orders,99441,99441
1,payments,103886,99440


promt: for tables order and payments show number of unique order_id

## 2. Plan

*Before you write the main code, write your plan in your own words: what one row of your final table represents, which tables you will join and in what order, how many rows you expect at each step, and how you will handle the problems you found.*

## 3. Build

*Build as far as you can in the time. Short, re-runnable cells are better than one long cell.*

## 4. Check

*Show how you know your answer is right. What did you compare, count or spot check, and what did the checks show? If a check failed, say what you changed.*

## 5. Prompts and AI notes

*Paste each prompt you used, as you sent it. Then note what you accepted, what you changed or rejected, and why. If applicable, include at least one example where the AI was wrong or needed correcting.*

## 6. What I could not verify

*Be honest about what remains unchecked, and what you would do next with more time.*